In [12]:
import pandas as pd
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt
import streamlit as st


df=yf.download('EURUSD=X', start='2020-08-01')
#On suppose que :
#-les frais d'achats et de ventes sont similaires
#-le slippage n'est pas réaliste

[*********************100%***********************]  1 of 1 completed


In [26]:
def strategy(df, window=1,fee=0.0001):
    df=df.copy()  # On fait une copie du DataFrame pour éviter de modifier l'original 
    df['ret']=np.log(df.Close.pct_change() + 1) # Calcul des rendements logarithmiques :pct_change() donne le rendement simple
    
    df['prior_n']=df.ret.rolling(window).sum() # somme des rendements passés (rolling window)
   
    df['position']=[1 if i>0 else -1 for i in df.prior_n]  # Création du signal de trading :+1 si le momentum est positif (on achète), -1 sinon (on vend / short)
    df['trade'] = df['position'].diff().abs() # Variation de position (détecte les trades)
    df['costs'] = df['trade'] * fee # on paye fee à chaque changement de position (passage long, short)
    df['strat']=df.position.shift(1)*df['ret'] #Rendement stratégie brut avec position décalée d'une période (on ne peut pas trader dans le passé)
   
    df['strat_net'] = df['strat'] - df['costs'] # On retire les frais
    df['equity_ret'] = df['strat_net'].cumsum()# Calcul des performances cumulées 
    df.dropna(inplace=True)  # Suppression des lignes avec NaN (au début à cause de pct_change et rolling)
   
    return df



In [27]:

strategy(df, 10)


Price,Close,High,Low,Open,Volume,ret,prior_n,position,trade,costs,strat,strat_net,equity_ret
Ticker,EURUSD=X,EURUSD=X,EURUSD=X,EURUSD=X,EURUSD=X,,,,,,,,
Date,,,,,,,,,,,,,
2020-08-17,1.185255,1.188072,1.183040,1.185045,0,0.003042,0.005944,1,2.0,0.0002,-0.003042,-0.003242,-0.006144
2020-08-18,1.187648,1.196516,1.187536,1.187931,0,0.002017,0.009574,1,0.0,0.0000,0.002017,0.002017,-0.004127
2020-08-19,1.194786,1.195457,1.189343,1.194729,0,0.005992,0.011145,1,0.0,0.0000,0.005992,0.005992,0.001865
2020-08-20,1.184350,1.186845,1.180652,1.184175,0,-0.008773,-0.002472,-1,2.0,0.0002,-0.008773,-0.008973,-0.007108
2020-08-21,1.187366,1.188637,1.175490,1.187085,0,0.002544,-0.000237,-1,0.0,0.0000,-0.002544,-0.002544,-0.009652
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-30,1.149201,1.152339,1.144387,1.149188,0,-0.003742,0.004964,1,0.0,0.0000,-0.003742,-0.003742,-0.152685
2026-03-31,1.146000,1.153882,1.144925,1.145790,0,-0.002789,-0.003444,-1,2.0,0.0002,-0.002789,-0.002989,-0.155674


In [18]:
df2=strategy(df)
    
def sharpe_ratio(df2,rf=0.0):
    r = df2['strat_net'].dropna() 
    if r.std() == 0:
        return 0
    return np.sqrt(252) * (r.mean()-rf) / r.std() #on utilise le sharp ratio annuel
   
sharpe_ratio(df2,0.0)
    
    

np.float64(-0.11884290166403155)

In [20]:
def optimize_window(df2, windows=range(1, 51),fee=0.0001):
    results = []
    for w in windows:
        df_temp = strategy(df2, window=w,fee=fee) #On applique la stratégie sur une copie indépendante
        results.append((w, sharpe_ratio(df_temp))) #Calcul du Sharpe ratio + stockage du résultat
    return results
#on split les data pour éviter l'overfitting
split = int(len(df2) * 0.7)
df_train = df2.iloc[:split]
df_test = df2.iloc[split:]
#Optimisation de la data train
results = optimize_window(df_train, windows=range(1, 51),fee=0.0001)
best_window = max(results, key=lambda x: x[1])[0]
best_sharpe = max(results, key=lambda x: x[1])[1]

print("Best window (train):", best_window)
print("Best Sharpe (train):", best_sharpe)

#Test sur TEST (out-of-sample)
df_test_strat = strategy(df_test, window=best_window)

test_sharpe = sharpe_ratio(df_test_strat)

print("Out-of-sample Sharpe:", test_sharpe)


    

Best window (train): 14
Best Sharpe (train): 0.5205448964919772
Out-of-sample Sharpe: -0.6462642051049067


In [10]:
import streamlit as st
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def strategy(df, window=1,fee=0.0001):
    df=df.copy()  # On fait une copie du DataFrame pour éviter de modifier l'original 
    df['ret']=np.log(df.Close.pct_change() + 1) # Calcul des rendements logarithmiques :pct_change() donne le rendement simple
    
    df['prior_n']=df.ret.rolling(window).sum() # somme des rendements passés (rolling window)
   
    df['position']=[1 if i>0 else -1 for i in df.prior_n]  # Création du signal de trading :+1 si le momentum est positif (on achète), -1 sinon (on vend / short)
    df['trade'] = df['position'].diff().abs() # Variation de position (détecte les trades)
    df['costs'] = df['trade'] * fee # on paye fee à chaque changement de position (passage long, short)
    df['strat']=df.position.shift(1)*df['ret'] #Rendement stratégie brut avec position décalée d'une période (on ne peut pas trader dans le passé)
   
    df['strat_net'] = df['strat'] - df['costs'] # On retire les frais
    df['equity_ret'] = df['strat_net'].cumsum()# Calcul des performances cumulées 
    df.dropna(inplace=True)  # Suppression des lignes avec NaN (au début à cause de pct_change et rolling)
   
    return df

def optimize_window(df2, windows=range(1, 51),fee=0.0001):
    results = []
    for w in windows:
        df_temp = strategy(df2, window=w,fee=fee) #On applique la stratégie sur une copie indépendante
        results.append((w, sharpe_ratio(df_temp))) #Calcul du Sharpe ratio + stockage du résultat
    return results

def sharpe_ratio(df2,rf=0.0):
    r = df2['strat_net'].dropna() 
    if r.std() == 0:
        return 0
    return np.sqrt(252) * (r.mean()-rf) / r.std() #on utilise le sharp ratio annuel sur les rendements net de frais 
#Création du Dash Board
# Sélection du ticker
TICKERS = {
    "EUR/USD"        : "EURUSD=X",
    "BTC/USD"        : "BTC-USD",
    "S&P 500"        : "^GSPC",
    "Gold"           : "GC=F",
    "Apple"          : "AAPL",
    "Nasdaq"         : "^IXIC",
}

st.title("📈 Quant Strategy Dashboard")

ticker_label = st.selectbox("Indice / Actif", options=list(TICKERS.keys()))
ticker       = TICKERS[ticker_label]

# Sélection de la période

col_d1, col_d2 = st.columns(2)
start_date = col_d1.date_input("Date de début", value=pd.Timestamp("2020-08-01"))
end_date   = col_d2.date_input("Date de fin",   value=pd.Timestamp("today"))

if start_date >= end_date:
    st.error("La date de début doit être antérieure à la date de fin.")
    st.stop()

# Chargement des données (mis en cache pour éviter de re-télécharger à chaque interaction)

@st.cache_data
def load_data(ticker, start, end):
    df = yf.download(ticker, start=start_date, end=end_date)
    if df.empty:
        st.error(f"Impossible de télécharger les données pour {ticker}.")
        st.stop()
    return df

df = load_data(ticker, start=start_date, end=end_date)

# Split train/test une seule fois, en dehors des fonctions

split = int(len(df) * 0.7)
df_train = df.iloc[:split]
df_test  = df.iloc[split:]


# Dates affichées pour que l'utilisateur sache sur quelle période il travaille
col1, col2 = st.columns(2)
col1.caption(f"Train : {df_train.index[0].date()} → {df_train.index[-1].date()}")
col2.caption(f"Test  : {df_test.index[0].date()}  → {df_test.index[-1].date()}")


# Paramètres utilisateur

st.subheader("Paramètres")
window = st.slider("Window (momentum)", 1, 50, 10)
fee = st.slider("Transaction cost (par trade)", min_value=0.0, max_value=0.001, value=0.0001, step=0.00001,format="%.5f")


# Appliquer la stratégie sur le dataset complet, puis séparer
# (évite le plateau au démarrage du test dû au rolling "froid")

df_full_strat  = strategy(df, window=window, fee=fee)

split_idx      = df_full_strat.index.get_loc(df_train.index[-1])
df_train_strat = df_full_strat.iloc[:split_idx+1]
df_test_strat  = df_full_strat.iloc[split_idx+1:]

# Métriques Sharpe — train vs test côte à côte
# (permet de détecter l'overfitting grâce à la différence des Sharpe ratios)

sr_train = sharpe_ratio(df_train_strat)
sr_test  = sharpe_ratio(df_test_strat)

st.subheader("Performance")
m1, m2 = st.columns(2)
m1.metric("Sharpe — Train",        round(sr_train, 3))
m2.metric("Sharpe — Out-of-sample", round(sr_test,  3),
          delta=round(sr_test - sr_train, 3))   # delta montre la dégradation éventuelle


# Equity curves — train ET test sur le même graphique
# On concatène les deux périodes pour un affichage continu

st.subheader("Equity curve")

def equity_curve(df_strat):
    return np.exp(df_strat[['ret', 'strat_net']].cumsum())

eq_train = equity_curve(df_train_strat)
eq_test  = equity_curve(df_test_strat)

# Recaler le test pour qu'il parte de la dernière valeur du train (courbe continue)
offset = eq_train.iloc[-1] - eq_test.iloc[0]
eq_test_adj = eq_test + offset

equity_full = pd.concat([eq_train, eq_test_adj])
equity_full.columns = ['Buy & Hold', 'Strategy']

fig, ax = plt.subplots(figsize=(10, 4))
equity_full['Buy & Hold'].plot(ax=ax, label='Buy & Hold', color='steelblue', linewidth=1)
equity_full['Strategy'].plot(ax=ax,   label='Strategy',   color='darkorange', linewidth=1)

# Ligne verticale pour marquer la séparation train/test
ax.axvline(df_train_strat.index[-1], color='gray', linestyle='--', linewidth=0.8, label='Train / Test')
ax.legend(fontsize=9)
ax.set_title("Equity curve — train + out-of-sample", fontsize=11)
ax.set_ylabel("Valeur du portefeuille (base 1)")
st.pyplot(fig)
plt.close(fig)   # libère la mémoire matplotlib


# Optimisation — uniquement sur le train, jamais sur df complet

st.subheader("Optimisation de la fenêtre")

if st.button("Lancer l'optimisation (train uniquement)"):
    with st.spinner("Calcul en cours..."):
        results = optimize_window(df_train, windows=range(1, 51), fee=fee)

    best_window = max(results, key=lambda x: x[1])[0]
    best_sharpe = max(results, key=lambda x: x[1])[1]

    # Vérification out-of-sample avec la meilleure fenêtre trouvée
    df_best_test = strategy(df_test, window=best_window, fee=fee)
    oos_sharpe   = sharpe_ratio(df_best_test)

    b1, b2, b3 = st.columns(3)
    b1.metric("Meilleure fenêtre",    best_window)
    b2.metric("Sharpe train optimal", round(best_sharpe, 3))
    b3.metric("Sharpe Out Of Sample",   round(oos_sharpe,  3),
              delta=round(oos_sharpe - best_sharpe, 3))

    # Graphique Sharpe vs Window
    windows_list = [w for w, s in results]
    sharpes_list = [s for w, s in results]

    fig2, ax2 = plt.subplots(figsize=(8, 3))
    ax2.plot(windows_list, sharpes_list, color='steelblue', linewidth=1.2)
    ax2.axvline(best_window, color='darkorange', linestyle='--', linewidth=1, label=f'Best = {best_window}')
    ax2.axhline(0, color='gray', linewidth=0.5)
    ax2.set_xlabel("Window")
    ax2.set_ylabel("Sharpe ratio")
    ax2.set_title("Sharpe ratio vs fenêtre de momentum (train)", fontsize=11)
    ax2.legend(fontsize=9)
    st.pyplot(fig2)
    plt.close(fig2)

2026-04-04 10:42:24.581 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-04 10:42:25.113 
  command:

    streamlit run C:\Users\illie\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-04-04 10:42:25.119 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-04 10:42:25.124 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-04 10:42:25.129 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-04 10:42:25.137 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-04 10:42:25.139 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-04 10:42: